In [49]:
!pip install groq gradio pandas matplotlib requests -q

In [50]:
from groq import Groq

client = Groq(
    api_key="YOUR_GROQ_API_KEY"
)

In [51]:
import pandas as pd
import requests
import io

url = "https://raw.githubusercontent.com/plotly/datasets/master/2011_us_ag_exports.csv"

df = pd.read_csv(url)

df.head()

,code,state,category,total exports,beef,pork,poultry,dairy,fruits fresh,fruits proc,total fruits,veggies fresh,veggies proc,total veggies,corn,wheat,cotton
0,AL,Alabama,state,1390.63,34.4,10.6,481.0,4.06,8.0,17.1,25.11,5.5,8.9,14.33,34.9,70.0,317.61
1,AK,Alaska,state,13.31,0.2,0.1,0.0,0.19,0.0,0.0,0.00,0.6,1.0,1.56,0.0,0.0,0.00
2,AZ,Arizona,state,1463.17,71.3,17.9,0.0,105.48,19.3,41.0,60.27,147.5,239.4,386.91,7.3,48.7,423.95
3,AR,Arkansas,state,3586.02,53.2,29.4,562.9,3.53,2.2,4.7,6.88,4.4,7.1,11.45,69.5,114.5,665.44
4,CA,California,state,16472.88,228.7,11.1,225.4,929.95,2791.8,5944.6,8736.40,803.2,1303.5,2106.79,34.6,249.3,1064.95


In [55]:
from google.colab import files

uploaded = files.upload()

Saving crop_production.csv to crop_production (2).csv


In [57]:
import pandas as pd

df = pd.read_csv("crop_production.csv")

print(df.columns)
print(df.shape)

df.head()

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'Production'],
      dtype='object')
(246091, 7)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [58]:
# Basic cleaning

df.columns = df.columns.str.strip()

df["State_Name"] = df["State_Name"].astype(str).str.strip()
df["District_Name"] = df["District_Name"].astype(str).str.strip()
df["Season"] = df["Season"].astype(str).str.strip()
df["Crop"] = df["Crop"].astype(str).str.strip()

df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")
df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

df = df.dropna(subset=["Production"])

print("✅ Cleaned")
print(df.shape)

✅ Cleaned
(242361, 7)


In [59]:
states = sorted(df["State_Name"].dropna().unique().tolist())
crops = sorted(df["Crop"].dropna().unique().tolist())

print("States:", len(states))
print("Crops:", len(crops))

print(states[:10])
print(crops[:10])

States: 33
Crops: 124
['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Goa', 'Gujarat']
['Apple', 'Arcanut (Processed)', 'Arecanut', 'Arhar/Tur', 'Ash Gourd', 'Atcanut (Raw)', 'Bajra', 'Banana', 'Barley', 'Bean']


In [60]:
import json

def interpret_question(question):

    prompt = f"""
You are a data analyst.

Dataset columns:
- State_Name
- District_Name
- Crop_Year
- Season
- Crop
- Area
- Production

Available states:
{states}

Available crops:
{crops[:50]}

Convert the user question into JSON.

Supported operations:

1. top_state
Example:
{{
  "operation":"top_state",
  "crop":"Rice",
  "year":2010
}}

2. trend
Example:
{{
  "operation":"trend",
  "crop":"Rice",
  "state":"Punjab"
}}

3. compare_states
Example:
{{
  "operation":"compare_states",
  "crop":"Rice",
  "year":2010,
  "states":["Punjab","Haryana"]
}}

4. top_n
Example:
{{
  "operation":"top_n",
  "crop":"Rice",
  "year":2010,
  "n":5
}}

5. summary
Example:
{{
  "operation":"summary",
  "crop":"Rice",
  "year":2010,
  "state":"Punjab"
}}

6. out_of_scope
Example:
{{
  "operation":"out_of_scope"
}}

Return ONLY JSON.
Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    text = response.choices[0].message.content

    # Remove markdown code fences
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    return json.loads(text)




In [61]:
def execute_query(query):

    operation = query.get("operation")

    # ==================================
    # OUT OF SCOPE
    # ==================================
    if operation == "out_of_scope":

        return {
            "answer": "This question cannot be answered using the crop production dataset."
        }

    # ==================================
    # TOP STATE
    # ==================================
    if operation == "top_state":

        crop = query["crop"]
        year = query["year"]

        filtered = df[
            (df["Crop"].str.lower() == crop.lower()) &
            (df["Crop_Year"] == year)
        ]

        grouped = (
            filtered
            .groupby("State_Name")["Production"]
            .sum()
            .reset_index()
        )

        grouped = grouped.sort_values(
            "Production",
            ascending=False
        )

        winner = grouped.iloc[0]

        return {
            "answer":
            f"{winner['State_Name']} produced the most {crop} in {year} "
            f"with production of {winner['Production']:,.0f}"
        }

    # ==================================
    # TREND
    # ==================================
    if operation == "trend":

        crop = query["crop"]
        state = query["state"]

        filtered = df[
            (df["Crop"].str.lower() == crop.lower()) &
            (df["State_Name"].str.lower() == state.lower())
        ]

        trend = (
            filtered
            .groupby("Crop_Year")["Production"]
            .sum()
            .reset_index()
            .sort_values("Crop_Year")
        )

        return {
            "answer":
            f"Showing {crop} production trend in {state}",
            "data": trend
        }

    # ==================================
    # TOP N STATES
    # ==================================
    if operation == "top_n":

        crop = query["crop"]
        year = query["year"]
        n = query["n"]

        filtered = df[
            (df["Crop"].str.lower() == crop.lower()) &
            (df["Crop_Year"] == year)
        ]

        grouped = (
            filtered
            .groupby("State_Name")["Production"]
            .sum()
            .reset_index()
        )

        top_n = (
            grouped
            .sort_values("Production", ascending=False)
            .head(n)
        )

        answer = f"Top {n} {crop} producing states in {year}\n\n"

        for idx, row in enumerate(top_n.itertuples(), start=1):

            answer += (
                f"{idx}. {row.State_Name} "
                f"({row.Production:,.0f})\n"
            )

        return {
            "answer": answer,
            "data": top_n
        }

    # ==================================
    # COMPARE STATES
    # ==================================
    if operation == "compare_states":

        crop = query["crop"]
        year = query["year"]
        states_list = query["states"]

        filtered = df[
            (df["Crop"].str.lower() == crop.lower()) &
            (df["Crop_Year"] == year) &
            (df["State_Name"].isin(states_list))
        ]

        comparison = (
            filtered
            .groupby("State_Name")["Production"]
            .sum()
            .reset_index()
        )

        answer = f"{crop} production comparison in {year}\n\n"

        for row in comparison.itertuples():
            answer += (
                f"{row.State_Name}: "
                f"{row.Production:,.0f}\n"
            )

        return {
            "answer": answer,
            "data": comparison
        }

    # ==================================
    # SUMMARY
    # ==================================
    if operation == "summary":

        crop = query["crop"]
        year = query["year"]
        state = query["state"]

        filtered = df[
            (df["Crop"].str.lower() == crop.lower()) &
            (df["Crop_Year"] == year) &
            (df["State_Name"].str.lower() == state.lower())
        ]

        total_area = filtered["Area"].sum()
        total_production = filtered["Production"].sum()

        return {
            "answer":
            f"{state} produced {total_production:,.0f} units of "
            f"{crop} in {year} across {total_area:,.0f} area."
        }

    # ==================================
    # FALLBACK
    # ==================================
    return {
        "answer": "Unsupported operation."
    }

In [62]:
import matplotlib.pyplot as plt

def create_chart(result):

    if "data" not in result:
        return None

    data = result["data"]

    plt.figure(figsize=(8,4))

    # TREND CHART
    if "Crop_Year" in data.columns:

        plt.plot(
            data["Crop_Year"],
            data["Production"],
            marker="o"
        )

        plt.xlabel("Year")
        plt.ylabel("Production")
        plt.title("Production Trend")

    # TOP N CHART
    elif "State_Name" in data.columns:

        plt.bar(
            data["State_Name"],
            data["Production"]
        )

        plt.xticks(rotation=45)
        plt.ylabel("Production")
        plt.title("Top States")

    plt.tight_layout()

    return plt.gcf()

In [63]:
import gradio as gr

def ask_question(question):

    try:
        query = interpret_question(question)

        result = execute_query(query)

        fig = create_chart(result)

        return result["answer"], fig

    except Exception as e:
        return f"Error: {str(e)}", None


demo = gr.Interface(
    fn=ask_question,
    inputs=gr.Textbox(
        label="Ask about crop production data"
    ),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Plot(label="Chart")
    ],
    title="NeuralCity GovInsight",
    description="Talk to Government Crop Production Data"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7f495650d637f3aa6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
